# 🌀 Notebook 1 — The Saga Pattern: Why It Exists

You're building an online shop. One **checkout** action must:

1. 📦 reserve stock in the `inventory` service
2. 💳 charge the customer in the `payment` service
3. 🚚 book a courier in the `shipping` service

Each service has **its own database**. They don't share tables, and nothing like `BEGIN TRANSACTION … COMMIT`
can span all three. So what happens if step 2 succeeds but step 3 fails? The customer was charged but
will never receive the item.

This notebook shows:

1. ❌ the **naive** way (no compensation) — and why it corrupts your system
2. ✅ the **saga** way — a sequence of local transactions where each step has an *undo*
3. the vocabulary you'll hear in real designs: *local transaction*, *compensating transaction*,
   *forward* vs *backward* recovery

> **Key idea.** A saga does not *rollback* — it *compensates*. Rollback erases history.
> Compensation adds a new step that reverses the business effect of an earlier step.


## 🛠️ Setup

```bash
cd 05-microservices/saga
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. ❌ Naive approach: call services in sequence, hope for the best

Below we simulate three services with tiny in-memory dictionaries.
The checkout function just calls them one after another. If the last one explodes,
we leave the system in a **broken** state: stock is gone and the customer is charged.


In [ ]:
# In-memory "databases" for three services
inventory = {"widget": 10}
payments  = {"total_charged": 0}
shipping  = {"bookings": []}

def reserve_stock(item):
    inventory[item] -= 1
    print(f"  inventory: reserved 1 {item} (left={inventory[item]})")

def charge(amount):
    payments["total_charged"] += amount
    print(f"  payment:   charged ${amount} (total={payments['total_charged']})")

def book_courier(item):
    # 💥 Simulate a crash: courier API is down today
    raise RuntimeError("courier API timeout")

def naive_checkout(item, amount):
    reserve_stock(item)
    charge(amount)
    book_courier(item)     # boom

try:
    naive_checkout("widget", 20)
except Exception as e:
    print(f"✗ checkout failed: {e}")

print()
print("👉 What state are we in now?")
print("   inventory:", inventory)
print("   payments :", payments)
print("   shipping :", shipping)
print()
print("The customer was charged $20 but will NEVER get the widget.")
print("Stock also shows 1 fewer unit even though no order actually completed.")


## 2. ✅ Saga approach: every step has an *undo*

A **saga** is a list of steps, where each step has:

| part | meaning | example |
|---|---|---|
| `do`   | the forward action (a local DB transaction in one service) | `charge($20)` |
| `undo` | the **compensating transaction** that reverses the business effect | `refund($20)` |

When a step fails, we **walk back through the already-completed steps** and run each `undo` in reverse order.
The system ends up in a consistent state — maybe not identical to the start (a refund is visible in
the ledger, for example), but **semantically correct**.

Below is a tiny `Saga` runner. Read it — it's only ~20 lines.


In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class Step:
    name: str
    do: Callable[[], None]
    undo: Callable[[], None]

class Saga:
    """Run steps in order. If any step fails, compensate the completed ones in reverse."""
    def __init__(self, steps: list[Step]):
        self.steps = steps

    def run(self) -> str:
        done: list[Step] = []
        for s in self.steps:
            try:
                print(f'→ {s.name}')
                s.do()
                done.append(s)
            except Exception as e:
                print(f"✗ step '{s.name}' failed: {e}")
                for d in reversed(done):
                    print(f"  ↶ compensating '{d.name}'")
                    d.undo()
                return 'compensated'
        return 'committed'


### Now re-run the e-commerce checkout — this time as a saga

Same three services, same failure in `book_courier`. But now each step has an `undo`,
so when shipping blows up we automatically refund the charge and return stock to inventory.


In [ ]:
# Reset the "databases"
inventory = {"widget": 10}
payments  = {"total_charged": 0, "refunds": 0}
shipping  = {"bookings": []}

# --- forward actions ---
def reserve_stock():  inventory["widget"] -= 1
def charge():         payments["total_charged"] += 20
def book_courier():   raise RuntimeError("courier API timeout")  # 💥

# --- compensating actions ---
def release_stock():  inventory["widget"] += 1
def refund():         payments["refunds"] += 20
def cancel_courier(): pass   # nothing to undo — shipping never booked

saga = Saga([
    Step("reserve_stock", reserve_stock, release_stock),
    Step("charge",        charge,        refund),
    Step("book_courier",  book_courier,  cancel_courier),
])

result = saga.run()

print()
print("result   :", result)
print("inventory:", inventory)
print("payments :", payments)
print("shipping :", shipping)
print()
print("✅ Stock is back to 10. The $20 charge was refunded. No phantom order.")


### ⚠️ The bug hiding in that runner: *what if an `undo` fails?*

Look at the compensation loop again. `d.undo()` is called with no protection. If the
refund service happens to be down at that exact moment, the exception escapes `run()`
and **the remaining compensations never execute** — so the stock is never released.

This is the single most common saga defect in the wild, because it only shows up on
the unhappy path *of* the unhappy path. Let's watch it happen.

In [ ]:
inventory = {"widget": 10}
payments  = {"total_charged": 0, "refunds": 0}

def reserve_stock():  inventory["widget"] -= 1
def charge():         payments["total_charged"] += 20
def book_courier():   raise RuntimeError("courier API timeout")

def release_stock():  inventory["widget"] += 1
def refund_broken():  raise RuntimeError("payment gateway unreachable")   # 💥
def cancel_courier(): pass

try:
    Saga([
        Step("reserve_stock", reserve_stock, release_stock),
        Step("charge",        charge,        refund_broken),
        Step("book_courier",  book_courier,  cancel_courier),
    ]).run()
except Exception as e:
    print(f"\n💥 the compensation loop itself blew up: {e}")

print()
print("inventory:", inventory, "← stock NEVER released: release_stock was never reached")
print("payments :", payments,  "← charge NEVER refunded either")
print()
print("The customer is charged, the stock is gone, and the saga reported nothing")
print("useful. This is strictly worse than the naive version, because we now")
print("*believe* we have compensation.")

### ✅ A runner that survives failing compensations

Three rules, all visible in the code below:

1. **Never let one failed `undo` skip the others.** Wrap each in its own `try`.
2. **Retry compensations.** They fail for the same transient reasons forward steps do,
   and unlike forward steps you can't just give up — the business state is already
   inconsistent.
3. **Escalate what you can't fix.** A compensation that permanently fails is not a
   log line, it's a **dead-letter entry plus a page**. A human has to reconcile it.

In [ ]:
import time

def compensate_with_retry(step, tries=3, base=0.02):
    """Retry one compensation. Returns True if it eventually succeeded."""
    for attempt in range(1, tries + 1):
        try:
            step.undo()
            return True
        except Exception as e:
            if attempt == tries:
                print(f"     ✗ '{step.name}' undo failed permanently: {e}")
                return False
            print(f"     … '{step.name}' undo attempt {attempt} failed ({e}); retrying")
            time.sleep(base * (2 ** (attempt - 1)))

class RobustSaga(Saga):
    def run(self) -> str:
        done: list[Step] = []
        for s in self.steps:
            try:
                print(f'→ {s.name}')
                s.do()
                done.append(s)
            except Exception as e:
                print(f"✗ step '{s.name}' failed: {e}")
                stuck = []
                for d in reversed(done):
                    print(f"  ↶ compensating '{d.name}'")
                    if not compensate_with_retry(d):
                        stuck.append(d.name)
                    # NOTE: we keep going either way — the other compensations
                    # must still run, whatever happened to this one.
                if stuck:
                    DEAD_LETTER.append({'saga': 'checkout', 'stuck_steps': stuck})
                    print(f"  🚨 escalated to humans: {stuck} could not be undone")
                    return 'compensated_with_failures'
                return 'compensated'
        return 'committed'

DEAD_LETTER = []

# Same broken refund as before — but now the stock still gets released.
inventory = {"widget": 10}
payments  = {"total_charged": 0, "refunds": 0}

result = RobustSaga([
    Step("reserve_stock", reserve_stock, release_stock),
    Step("charge",        charge,        refund_broken),
    Step("book_courier",  book_courier,  cancel_courier),
]).run()

print()
print("result   :", result)
print("inventory:", inventory, "← released ✅ (one broken undo no longer blocks the rest)")
print("payments :", payments,  "← still not refunded ❌ — but we KNOW about it:")
print("dead letter queue:", DEAD_LETTER)

> **The honest lesson:** a saga cannot guarantee consistency, only *convergence*. When
> a compensation is permanently impossible, the correct outcome is a **loud, durable
> record for a human**, not a swallowed exception. Every production saga engine
> (Temporal, Step Functions, Camunda) exposes exactly this: a stuck-workflow list that
> someone is on the hook for draining.

### 🧭 When do I need a saga — and when should I not?

**✅ Use one when** a single business operation writes to **more than one service's
database** and you need all-or-nothing *business* semantics — checkout, booking,
onboarding, order fulfilment. Also when the flow is **long-running** (minutes to days,
waiting on a human or a courier), where holding a distributed lock is out of the
question.

**🚫 Don't** when:

- **It all fits in one service.** A local ACID transaction is simpler, stronger, and
  free. Splitting a transaction across services to "be microservicey" and then bolting a
  saga on top is a self-inflicted wound. Look hard at whether the services are drawn in
  the wrong place first.
- **You can't write a compensation.** If a step is irreversible and there is nothing
  semantically equivalent to undoing it, a saga gives you the illusion of safety.
  Reorder the steps so the irreversible one is last (the *pivot*), or don't use a saga.
- **The read-side must never see intermediate state.** Sagas have **no isolation**: other
  transactions *will* observe the half-finished state between steps. If "briefly reserved
  then released" is unacceptable to the business, you need a different design (a
  reservation/pending status the UI understands, or a single-service transaction).
- **You'd hand-roll the durability.** If the flow matters, use Temporal / Step Functions /
  Camunda rather than reimplementing notebook 4 under deadline pressure.

## ✅ Recap

- Distributed transactions across services **cannot** use classic 2-phase commit at web scale.
- A **saga** is a sequence of local transactions. If any step fails, previous steps are **compensated**.
- Compensation ≠ rollback: it's a new forward action that cancels the earlier business effect.
- Next notebook: **how** to coordinate these steps — with a central brain (orchestration) or by having
  services listen to each other's events (choreography).
